In [2]:
import pandas as pd
import numpy as np
# from sqlalchemy import create_engine

In [11]:
def load_stats(): # function for loading player stats
    batting = pd.read_csv('data/bwar_bat.csv')
    pitching = pd.read_csv('data/bwar_pitch.csv')
    hof = pd.read_csv('data/hof_players.csv')
    
    return batting.head(), pitching.head(), hof.head()

In [14]:
batting_head, pitching_head, hof_head = load_stats()

In [15]:
batting_head

,name_common,mlb_ID,player_ID,year_ID,team_ID,stint_ID,lg_ID,pitcher,G,PA,salary,runs_above_avg,runs_above_avg_off,runs_above_avg_def,WAR_rep,WAA,WAR
0,David Aardsma,430911.0,aardsda01,2004,SFG,1,NL,Y,11,0.0,300000.0,0.0,0.0,0.0,0.0,0.00,0.00
1,David Aardsma,430911.0,aardsda01,2006,CHC,1,NL,Y,43,3.0,NaN,-0.4,-0.4,0.0,0.0,-0.04,-0.04
2,David Aardsma,430911.0,aardsda01,2007,CHW,1,AL,Y,2,0.0,387500.0,0.0,0.0,0.0,0.0,0.00,0.00
3,David Aardsma,430911.0,aardsda01,2008,BOS,1,AL,Y,5,1.0,403250.0,-0.2,-0.2,0.0,0.0,-0.02,-0.02
4,David Aardsma,430911.0,aardsda01,2009,SEA,1,AL,Y,3,0.0,419000.0,0.0,0.0,0.0,0.0,0.00,0.00


In [16]:
pitching_head

,name_common,mlb_ID,player_ID,year_ID,team_ID,stint_ID,lg_ID,G,GS,RA,xRA,BIP,BIP_perc,salary,ERA_plus,WAR_rep,WAA,WAA_adj,WAR
0,George Bechtel,110756.0,bechtge01,1871,ATH,1,NaN,3,3,42,29.183,134.0,0.1176,NaN,52.682609,0.2455,-0.5925,-0.0156,-0.36
1,Dick McBride,118523.0,mcbridi01,1871,ATH,1,NaN,25,25,223,249.177,1001.0,0.8788,NaN,90.425664,2.0901,1.7600,-0.1330,3.72
2,Levi Meyerle,119019.0,meyerle01,1871,ATH,1,NaN,1,0,1,1.122,4.0,0.0035,NaN,63.000000,0.0102,0.0078,-0.0006,0.02
3,Al Spalding,122558.0,spaldal01,1871,BOS,1,NaN,31,31,272,289.515,1216.0,0.9233,1500.0,127.121875,2.4385,1.3733,-0.1542,3.66
4,Harry Wright,124617.0,wrighha01,1871,BOS,1,NaN,9,0,31,21.001,101.0,0.0767,NaN,69.307692,0.1885,-0.5283,-0.0112,-0.35


In [17]:
hof_head

,Year,Player,mlb_id
0,2025,Billy Wagner,123790.0
1,2025,CC Sabathia,282332.0
2,2025,Dave Parker,120225.0
3,2025,Dick Allen,110157.0
4,2025,Ichiro Suzuki,400085.0


In [19]:
def feature_eng(batting, pitching, hof):
    features = [] # store stats
    for stats_df in [batting, pitching]:
        # group by player for career stats
        career_stats = stats_df.groupby('player_id').agg({ # group by player_id to get metrics
            # career span
            'WAR': ['mean','max','sum'], # 3 WAR stats for each player: avg, max, total
            'year': ['min','max'] # career timespan: first season, last season
        }).reset_index() # converts back to regular df

        # additional features
        career_stats['years_played'] = career_stats['year']['max'] - career_stats['year']['min'] # years_played = career length (last year - first year)
        career_stats['is_hof'] = career_stats['player_id'].isin(hof['mlb_id']) # binary label to check if player in hof or not using isin()

        features.append(career_stats) # add stats to features

    return pd.concat(features, ignore_index = True) 

In [ ]:
from sklearn.model_selection import train_test_split